# Build the curated analytics tableReads the raw Parquet mirror written by `Export_to_parquet.ipynb`, flattens thesingle annual statement out of `data[0]`, joins the register attributes, derivesoperating margin, and writes one row per registered entity as a single Parquetfile for PowerBI.**Scope.** Every entity in the register, not only those that filed. Financialcolumns are null where no statement exists, and `operating_margin_status` records`no_filing` for those rows. An inner join would have been simpler, but it woulddelete the entities that make the coverage gap visible, and the near-totalabsence of filings for ENK and foreninger is itself a result. All legal formsare kept and `organisasjonsform_kode` is a column, so an AS-only view is adownstream filter rather than a decision baked into the table.**Operating margin.** `(driftsresultat / sumDriftsinntekter) * 100`, in percent,stored unrounded. Null whenever the ratio is undefined or misleading, with thereason recorded in `operating_margin_status`. Precedence, first match wins:| status | condition ||---|---|| `no_filing` | no statement was retrieved for this entity || `revenue_missing` | `sumDriftsinntekter` absent || `revenue_zero` | revenue exactly 0 || `revenue_negative` | revenue < 0 || `income_missing` | revenue > 0 but `driftsresultat` absent || `computed` | everything else; margin is populated |The zero test is an exact floating-point comparison. That is safe here becausethe API reports whole NOK amounts, so an intended zero is stored as exactly`0.0` rather than as a rounding residue.**Two aggregation regimes.** Metrics built from the ratio — median, quantiles,the distribution, per-company comparison — use all rows, because a margin isscale-invariant and a USD filer's margin is as valid as a NOK filer's. Metricsthat sum money across companies — pooled margin, total revenue, total assets —must use NOK rows only, because adding USD to NOK adds unlike units. The`currency_comparable` flag marks the second set. Nothing is filtered out of thetable; the flag gates aggregates rather than rows.FX normalisation was considered and rejected: it would require a fourth datasource, it would demand a defensible choice between period-average and closingrates across filings with many different accounting periods, and it would notchange any per-company margin — only the weight foreign filers carry in apooled figure.**Numerator and denominator are kept as columns.** A group-level operatingmargin is `SUM(operating_income) / SUM(revenue)` — the pooled margin — which isnot the mean of the per-company margins. Given the skew, the two differsubstantially. Keeping the components lets PowerBI compute the pooled figurewith `DIVIDE(SUM(...), SUM(...))` while the row-level column servesdistributional analysis.The same principle governs every other ratio. Equity ratio and current ratio are**not** stored: their components are, and PowerBI should divide sums rather thanaverage stored quotients. Only operating margin gets a column, because only itcarries a status taxonomy explaining when it is undefined.**No winsorised column.** The margin is stored unbounded. A company with 5,000NOK revenue and a 2 MNOK operating loss yields -40,000% legitimately; that issignal about the register, not an error. Outlier handling belongs in theanalysis notebook, where it can be varied.**Comparability is flagged, not filtered.** `avviklingsregnskap` and`is_full_year` mark liquidation accounts and non-annual periods. Both stay inthe table.**Total columns, no tri-state booleans.** About 1,083 entities have no
`financial_data` document at all — the HTTP 500 population that is never
written. The left join leaves their `fetch_status` NULL, which would make
`har_regnskap` NULL rather than False and let a PowerBI filter on FALSE skip
them silently. Both columns are made total in the build: `fetch_status` gains a
third value, `no_document`, and `har_regnskap` is coalesced to a real boolean.

**Date casting.** Spark 4 enables ANSI mode by default, under which a malformed
string cast to DATE raises rather than returning null. The three register date
columns are cast with `try_cast` so one bad value cannot abort a 1.17M-row
build, and the date probe cell counts what that nulls so the tolerance is
measured rather than silent.

**Filesystem note.** `data/` is a bind mount onto a Dropbox-synced Windowsfolder. Spark's directory-based output is therefore staged on container-localdisk under `/tmp`, and only the finished single file is copied onto the mount.Writing a Spark output directory directly into `data/` fails on cleanup with`PermissionError` on `rmdir`, because `shutil.rmtree` uses file-descriptor-relative calls that the bind mount does not reliably support, and becauseDropbox may hold a handle on the directory. Pausing Dropbox sync before a runis still advisable.

In [1]:
import glob
import json
import os
import shutil
import statistics
import time
from datetime import datetime, timezone

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA_DIR = "/home/jovyan/data"
PARQUET_DIR = os.path.join(DATA_DIR, "parquet")

# Single file rather than a part-file directory, so PowerBI's Parquet connector
# takes a plain path and needs no Folder/Combine step.
ANALYTICS_FILE = os.path.join(PARQUET_DIR, "analytics_company_financials.parquet")

# All Spark directory output goes to container-local disk. data/ is a bind mount
# onto a Dropbox-synced Windows folder where removing a directory fails
# intermittently even though writing into it succeeds, so only finished single
# files are placed on the mount.
STAGING_DIR = "/tmp/group13_analytics_staging"
SCRATCH_DIR = "/tmp/group13_scalability"

spark = SparkSession.builder.appName("group13_build_analytics").getOrCreate()

# Every measurement the notebook makes is added to this dict as it is produced,
# and dumped to JSON by the last cell. Anything printed but not recorded here
# is lost the moment the notebook is re-run, so nothing that a report might
# cite should exist only as cell output.
summary = {
    "built_at": datetime.now(timezone.utc).isoformat(),
    # Captured now rather than at the end: the scalability experiment stops
    # this session, after which none of these are reachable.
    "spark_version": spark.version,
    "spark_master": spark.sparkContext.master,
    "driver_max_heap_gb": spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3,
    "default_parallelism": spark.sparkContext.defaultParallelism,
    "cpu_cores": os.cpu_count(),
}

for k, v in summary.items():
    print("%-22s %s" % (k, v))

built_at               2026-09-02T20:31:54.241109+00:00
spark_version          4.2.0
spark_master           local[4]
driver_max_heap_gb     8.0
default_parallelism    4
cpu_cores              12


## Date parse check

Run before the build. Spark 4 enables ANSI mode by default, and under it a
malformed string cast to DATE raises `CAST_INVALID_INPUT` rather than returning
null. The build casts three register date columns that were previously carried
as strings, so a single bad value in 1.17M rows would abort it.

The build uses `try_cast`, which nulls bad values instead of raising. That
removes the abort risk but would hide the problem, so this cell counts what
`try_cast` will null. Unparseable dates in a national register are a
data-quality finding in their own right, and the counts go to the summary
whether they are zero or not.

In [2]:
raw_companies = spark.read.parquet(os.path.join(PARQUET_DIR, "companies"))

DATE_COLUMNS = ["stiftelsesdato", "konkursdato", "registreringsdatoEnhetsregisteret"]

print("%-40s %10s %10s %10s" % ("column", "non-null", "unparseable", "pct"))
print("-" * 74)
date_parse = {}
for col in DATE_COLUMNS:
    n_present = raw_companies.filter(F.col(col).isNotNull()).count()
    n_bad = raw_companies.filter(
        F.col(col).isNotNull()
        & F.expr("try_cast(%s as date)" % col).isNull()).count()
    pct = 100.0 * n_bad / n_present if n_present else 0.0
    date_parse[col] = {"non_null": n_present, "unparseable": n_bad, "pct": pct}
    print("%-40s %10d %10d %9.4f%%" % (col, n_present, n_bad, pct))

summary["date_parse_check"] = date_parse

column                                     non-null unparseable        pct
--------------------------------------------------------------------------
stiftelsesdato                               675310          0    0.0000%
konkursdato                                    3194          0    0.0000%
registreringsdatoEnhetsregisteret           1171373          0    0.0000%


## The build

One function, so the scalability experiment further down runs exactly the code
that produces the deliverable rather than a simplified stand-in.

`data` is typed as an array because that is what the API returns, but the
profiling pass confirmed exactly one element in every populated record, so
element 0 is taken directly and the join stays one-to-one. No `explode`.Dates
arrive as strings and are cast, so PowerBI receives real date types rather than
text.

`stiftelsesaar` and `regnskapsaar` are derived here because a year is the
dimension most charts slice on, and deriving once in code is cheaper and less
error-prone than a Power Query custom column.

In [3]:
# try_cast rather than to_date for the register date columns. Spark 4 enables
# ANSI mode by default, where casting a malformed string to DATE raises
# CAST_INVALID_INPUT instead of returning null, so one bad value anywhere in the
# 1.17M register rows would abort the build. try_cast nulls it instead; the date
# probe cell above counts how many values that affects, so nothing is lost
# silently. The regnskapsperiode dates keep to_date: they are nested inside the
# statement struct, they have already run clean, and a SQL string expression
# over a nested path adds resolution risk for no gain.
def _date(col):
    return F.expr("try_cast(%s as date)" % col)


# Paths into the nested statement. Named once so a typo cannot silently produce
# a column of nulls in one place and not another.
_RES = "d.resultatregnskapResultat"
_DR = _RES + ".driftsresultat"
_FIN = _RES + ".finansresultat"
_BAL = "d.egenkapitalGjeld"


# Returns the curated DataFrame: one row per registered entity, financial
# columns null where nothing was filed. Nothing is cached and no action is
# triggered, so the caller decides when and how the work is materialised.
# Taking a session argument rather than closing over a global is what lets the
# scalability experiment re-run this exact code under a different master.
def build(session):
    companies = session.read.parquet(os.path.join(PARQUET_DIR, "companies")).select(
        "organisasjonsnummer",
        F.col("navn"),
        F.col("organisasjonsform.kode").alias("organisasjonsform_kode"),
        F.col("organisasjonsform.beskrivelse").alias("organisasjonsform_navn"),
        F.col("naeringskode1.kode").alias("naeringskode1_kode"),
        F.col("naeringskode1.beskrivelse").alias("naeringskode1_beskrivelse"),
        F.col("institusjonellSektorkode.kode").alias("sektorkode"),
        F.col("institusjonellSektorkode.beskrivelse").alias("sektor_navn"),
        F.col("forretningsadresse.kommunenummer").alias("kommunenummer"),
        F.col("forretningsadresse.kommune").alias("kommune"),
        F.col("forretningsadresse.poststed").alias("poststed"),
        F.col("forretningsadresse.landkode").alias("landkode"),
        F.col("antallAnsatte").alias("antall_ansatte"),
        _date("stiftelsesdato").alias("stiftelsesdato"),
        F.year(_date("stiftelsesdato")).alias("stiftelsesaar"),
        _date("registreringsdatoEnhetsregisteret").alias("registrert_dato"),
        F.col("konkurs"),
        _date("konkursdato").alias("konkursdato"),
        F.col("underAvvikling").alias("under_avvikling"),
        F.col("registrertIMvaregisteret").alias("mva_registrert"),
        F.col("erIKonsern").alias("i_konsern"),
        F.col("kapital.belop").alias("aksjekapital"),
        F.col("sisteInnsendteAarsregnskap").alias("siste_aarsregnskap"),
    )

    # No filter on fetch_status: rows without a filing are carried through so
    # the coverage gap stays visible. The statement struct is null for them.
    financial = (
        session.read.parquet(os.path.join(PARQUET_DIR, "financial_data"))
        .select("organisasjonsnummer", "fetch_status", F.col("data")[0].alias("d"))
    )

    revenue = F.col(_DR + ".driftsinntekter.sumDriftsinntekter")
    income = F.col(_DR + ".driftsresultat")
    filed = F.col("fetch_status") == "success"

    # A margin is only meaningful with a strictly positive denominator and a
    # present numerator. Anything else is null, with the cause recorded.
    computable = filed & revenue.isNotNull() & (revenue > 0) & income.isNotNull()

    status = (
        F.when(~filed | F.col("fetch_status").isNull(), "no_filing")
        .when(revenue.isNull(), "revenue_missing")
        .when(revenue == 0, "revenue_zero")
        .when(revenue < 0, "revenue_negative")
        .when(income.isNull(), "income_missing")
        .otherwise("computed")
    )

    margin = F.when(computable, (income / revenue) * 100).otherwise(
        F.lit(None).cast("double"))

    period_from = F.to_date(F.col("d.regnskapsperiode.fraDato"))
    period_to = F.to_date(F.col("d.regnskapsperiode.tilDato"))
    # Inclusive of both endpoints: 2025-01-01 to 2025-12-31 is 365 days.
    period_days = F.datediff(period_to, period_from) + 1

    return (
        companies.join(financial, "organisasjonsnummer", "left")
        .select(
            # Register identity and attributes
            "organisasjonsnummer", "navn",
            "organisasjonsform_kode", "organisasjonsform_navn",
            "naeringskode1_kode", "naeringskode1_beskrivelse",
            "sektorkode", "sektor_navn",
            "kommunenummer", "kommune", "poststed", "landkode",
            "antall_ansatte", "stiftelsesdato", "stiftelsesaar", "registrert_dato",
            "konkurs", "konkursdato", "under_avvikling",
            "mva_registrert", "i_konsern", "aksjekapital", "siste_aarsregnskap",

            # Filing metadata, including the comparability flags.
            # ~1,083 entities have no financial_data document at all: the HTTP
            # 500 population that is never written. After the left join their
            # fetch_status is NULL, so `filed` is NULL rather than False. Both
            # columns are made total here: har_regnskap becomes a real boolean
            # so a PowerBI filter on FALSE cannot silently skip them, and
            # fetch_status gains a third value so the API-failure population is
            # countable rather than buried in a null.
            F.coalesce(F.col("fetch_status"), F.lit("no_document")).alias("fetch_status"),
            F.coalesce(filed, F.lit(False)).alias("har_regnskap"),
            F.col("d.regnskapstype").alias("regnskapstype"),
            F.col("d.oppstillingsplan").alias("oppstillingsplan"),
            F.col("d.valuta").alias("valuta"),
            F.col("d.virksomhet.morselskap").alias("morselskap"),
            F.col("d.regnkapsprinsipper.smaaForetak").alias("smaa_foretak"),
            F.col("d.regnkapsprinsipper.regnskapsregler").alias("regnskapsregler"),
            F.col("d.avviklingsregnskap").alias("avviklingsregnskap"),
            F.col("d.revisjon.ikkeRevidertAarsregnskap").alias("ikke_revidert"),
            F.col("d.revisjon.fravalgRevisjon").alias("fravalg_revisjon"),
            period_from.alias("period_from"),
            period_to.alias("period_to"),
            period_days.alias("period_days"),
            (period_days.isin(365, 366)).alias("is_full_year"),
            F.year(period_to).alias("regnskapsaar"),
            # Money amounts may only be summed across rows sharing a currency.
            # The margin ratio is scale-invariant and needs no such restriction,
            # so this gates aggregates rather than rows: every filing keeps its
            # per-company margin, but only NOK filings enter a pooled figure.
            (F.col("d.valuta") == "NOK").alias("currency_comparable"),

            # Income statement
            revenue.alias("revenue"),
            F.col(_DR + ".driftskostnad.sumDriftskostnad").alias("operating_costs"),
            income.alias("operating_income"),
            F.col(_FIN + ".finansinntekt.sumFinansinntekter").alias("finansinntekter"),
            F.col(_FIN + ".finanskostnad.sumFinanskostnad").alias("finanskostnader"),
            F.col(_FIN + ".nettoFinans").alias("netto_finans"),
            F.col(_RES + ".ordinaertResultatFoerSkattekostnad").alias("resultat_foer_skatt"),
            F.col(_RES + ".aarsresultat").alias("aarsresultat"),
            F.col(_RES + ".totalresultat").alias("totalresultat"),

            # Balance sheet, assets
            F.col("d.eiendeler.sumEiendeler").alias("sum_eiendeler"),
            F.col("d.eiendeler.omloepsmidler.sumOmloepsmidler").alias("omloepsmidler"),
            F.col("d.eiendeler.anleggsmidler.sumAnleggsmidler").alias("anleggsmidler"),

            # Balance sheet, equity and liabilities
            F.col(_BAL + ".sumEgenkapitalGjeld").alias("sum_egenkapital_gjeld"),
            F.col(_BAL + ".egenkapital.sumEgenkapital").alias("sum_egenkapital"),
            F.col(_BAL + ".egenkapital.innskuttEgenkapital.sumInnskuttEgenkaptial").alias("innskutt_egenkapital"),
            F.col(_BAL + ".egenkapital.opptjentEgenkapital.sumOpptjentEgenkapital").alias("opptjent_egenkapital"),
            F.col(_BAL + ".gjeldOversikt.sumGjeld").alias("sum_gjeld"),
            F.col(_BAL + ".gjeldOversikt.kortsiktigGjeld.sumKortsiktigGjeld").alias("kortsiktig_gjeld"),
            F.col(_BAL + ".gjeldOversikt.langsiktigGjeld.sumLangsiktigGjeld").alias("langsiktig_gjeld"),

            # Derived
            margin.alias("operating_margin_pct"),
            status.alias("operating_margin_status"),
        )
    )

## Input profile

Run before the write. These are the frequencies that justify the null rules,
and they belong in the report — the size of each undefined-margin class is a
data-quality result, not a footnote. Counted over every row, not a sample.

In [4]:
curated = build(spark).cache()

# Both sides are measured rather than compared against a stored constant. Two
# properties are under test: the left join must preserve exactly one row per
# registered entity, and the filed subset must match the mirror exactly.
raw_comp = spark.read.parquet(os.path.join(PARQUET_DIR, "companies"))
raw_fin = spark.read.parquet(os.path.join(PARQUET_DIR, "financial_data"))

n_companies = raw_comp.count()
n_source_filings = raw_fin.filter("fetch_status = 'success'").count()
n_rows = curated.count()
n_distinct = curated.select("organisasjonsnummer").distinct().count()
n_filed = curated.filter("har_regnskap").count()

print("Entities in register         : %d" % n_companies)
print("Rows in curated table        : %d  %s"
      % (n_rows, "OK" if n_rows == n_companies else "MISMATCH"))
print("Distinct organisasjonsnummer : %d  %s"
      % (n_distinct, "OK" if n_distinct == n_rows else "DUPLICATES"))
print("Successful filings in mirror : %d" % n_source_filings)
print("Rows with a filing           : %d  %s"
      % (n_filed, "OK" if n_filed == n_source_filings else "MISMATCH"))
assert n_rows == n_companies, "left join changed cardinality"
assert n_distinct == n_rows, "duplicate organisasjonsnummer after join"
assert n_filed == n_source_filings, "filed subset does not match the mirror"

summary["cardinality"] = {"register_entities": n_companies, "rows": n_rows,
                          "distinct_organisasjonsnummer": n_distinct,
                          "source_filings": n_source_filings, "rows_filed": n_filed}

print("\nOperating margin status over all %d rows:" % n_rows)
status_rows = (curated.groupBy("operating_margin_status").count()
               .orderBy(F.desc("count")).collect())
for r in status_rows:
    print("  %-18s %8d  %6.2f%%"
          % (r["operating_margin_status"], r["count"], 100.0 * r["count"] / n_rows))
status_counts = {r["operating_margin_status"]: r["count"] for r in status_rows}

# Coverage by legal form. Reported, not asserted: this is the finding.
print("\nCoverage by legal form (top 12 by count):")
coverage_rows = (curated.groupBy("organisasjonsform_kode")
                 .agg(F.count(F.lit(1)).alias("antall"),
                      F.sum(F.col("har_regnskap").cast("int")).alias("med_regnskap"))
                 .orderBy(F.desc("antall")).limit(12).collect())
coverage = {}
for r in coverage_rows:
    pct = 100.0 * r["med_regnskap"] / r["antall"]
    coverage[r["organisasjonsform_kode"]] = {"antall": r["antall"],
                                             "med_regnskap": r["med_regnskap"],
                                             "dekning_pst": pct}
    print("  %-8s %9d entities  %9d filed  %6.2f%%"
          % (r["organisasjonsform_kode"], r["antall"], r["med_regnskap"], pct))

# stiftelsesdato is absent on a large minority of the register, so stiftelsesaar
# carries a null bucket and any founding-year chart silently drops those rows.
# Almost certainly this tracks legal form -- ENK and foreninger have no
# incorporation event in the same sense as an AS -- which would make it a
# property of the register rather than a defect. registrert_dato is populated
# everywhere and is the safer dimension for anything spanning all legal forms.
n_no_stift = curated.filter(F.col("stiftelsesdato").isNull()).count()
print("\nstiftelsesdato missing on %d of %d entities (%.2f%%), by legal form:"
      % (n_no_stift, n_rows, 100.0 * n_no_stift / n_rows))
stift_rows = (curated.groupBy("organisasjonsform_kode")
              .agg(F.count(F.lit(1)).alias("antall"),
                   F.sum(F.when(F.col("stiftelsesdato").isNull(), 1)
                          .otherwise(0)).alias("uten_stiftelsesdato"))
              .orderBy(F.desc("antall")).limit(12).collect())
stiftelsesdato_coverage = {}
for r in stift_rows:
    pct = 100.0 * r["uten_stiftelsesdato"] / r["antall"]
    stiftelsesdato_coverage[r["organisasjonsform_kode"]] = {
        "antall": r["antall"], "uten_stiftelsesdato": r["uten_stiftelsesdato"],
        "mangler_pst": pct}
    print("  %-8s %9d entities  %9d without  %6.2f%%"
          % (r["organisasjonsform_kode"], r["antall"], r["uten_stiftelsesdato"], pct))

# Flags are only meaningful on filed rows, so they are counted against that base.
print("\nComparability flags (share of the %d filed rows):" % n_filed)
flag_counts = {}
for col in ["avviklingsregnskap", "is_full_year", "morselskap", "smaa_foretak",
            "currency_comparable"]:
    flag_counts[col] = curated.filter(F.col(col)).count()
    print("  %-20s true in %8d  %6.2f%%"
          % (col, flag_counts[col], 100.0 * flag_counts[col] / n_filed))

print("\nCurrency (a ratio is scale-invariant, so this does not distort the margin):")
valuta_counts = {r["valuta"]: r["count"] for r in
                 curated.filter("har_regnskap").groupBy("valuta").count()
                 .orderBy(F.desc("count")).collect()}
for k, v in valuta_counts.items():
    print("  %-8s %8d" % (k, v))

# Three values now: success, no_data, and no_document for the entities that
# have no financial_data record at all (the HTTP 500 population).
print("\nFetch status across all %d entities:" % n_rows)
fetch_status_counts = {r["fetch_status"]: r["count"] for r in
                       curated.groupBy("fetch_status").count()
                       .orderBy(F.desc("count")).collect()}
for k, v in fetch_status_counts.items():
    print("  %-14s %9d  %6.2f%%" % (k, v, 100.0 * v / n_rows))

print("\nStatement type (KONSERN would mean consolidated figures alongside company ones):")
regnskapstype_counts = {r["regnskapstype"]: r["count"] for r in
                        curated.filter("har_regnskap").groupBy("regnskapstype").count()
                        .orderBy(F.desc("count")).collect()}
for k, v in regnskapstype_counts.items():
    print("  %-12s %8d" % (k, v))

summary["fetch_status_counts"] = fetch_status_counts
summary["status_counts"] = status_counts
summary["coverage_by_legal_form"] = coverage
summary["stiftelsesdato_missing_total"] = n_no_stift
summary["stiftelsesdato_coverage_by_legal_form"] = stiftelsesdato_coverage
summary["flag_counts"] = flag_counts
summary["valuta_counts"] = valuta_counts
summary["regnskapstype_counts"] = regnskapstype_counts

Entities in register         : 1171373
Rows in curated table        : 1171373  OK
Distinct organisasjonsnummer : 1171373  OK
Successful filings in mirror : 444645
Rows with a filing           : 444645  OK

Operating margin status over all 1171373 rows:
  no_filing            726728   62.04%
  computed             293155   25.03%
  revenue_missing       91026    7.77%
  revenue_zero          59206    5.05%
  revenue_negative        875    0.07%
  income_missing          383    0.03%

Coverage by legal form (top 12 by count):
  ENK         461154 entities       3266 filed    0.71%
  AS          431581 entities     403782 filed   93.56%
  FLI         128677 entities       2810 filed    2.18%
  ESEK         33212 entities      10324 filed   31.09%
  UTLA         28118 entities          0 filed    0.00%
  NUF          25838 entities       3545 filed   13.72%
  DA           10531 entities       1211 filed   11.50%
  BRL          10244 entities       9995 filed   97.57%
  KBO           7519 e

## Distribution of the computed margin

Reported with medians and percentiles rather than a mean. The margin is
unbounded below and heavily skewed, so a mean is dominated by a handful
ofmicroscopic-revenue companies and describes nothing.

The pooled margin below is the economically meaningful aggregate. The gap
between it and the median is the point worth discussing in the report.

In [5]:
computed = curated.filter("operating_margin_status = 'computed'")
n_computed = computed.count()

qs = computed.approxQuantile("operating_margin_pct",
                             [0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999],
                             0.0)
labels = ["p0.1", "p1", "p5", "p25", "median", "p75", "p95", "p99", "p99.9"]
print("Per-company operating margin (%), computed rows only:")
for lab, q in zip(labels, qs):
    print("  %-8s %14.2f" % (lab, q))

extremes = computed.agg(F.min("operating_margin_pct"), F.max("operating_margin_pct"),
                        F.mean("operating_margin_pct")).collect()[0]
print("  %-8s %14.2f" % ("min", extremes[0]))
print("  %-8s %14.2f" % ("max", extremes[1]))
print("  %-8s %14.2f   <- reported only to show why it is not used" % ("mean", extremes[2]))

pooled = computed.agg(F.sum("operating_income"), F.sum("revenue")).collect()[0]
pooled_pct = 100.0 * pooled[0] / pooled[1]
print("\nPooled margin  SUM(operating_income)/SUM(revenue)*100 = %.4f%%" % pooled_pct)

# The per-company margin is a ratio and therefore scale-invariant, so a filing
# reported in USD is as valid as one in NOK. The pooled margin is not: summing
# across currencies adds unlike units. Non-NOK filings are a fraction of a
# percent of the corpus, but that is a claim worth measuring rather than
# asserting, so the NOK-only figure is computed and compared.
nok = computed.filter("currency_comparable")
pooled_nok = nok.agg(F.sum("operating_income"), F.sum("revenue")).collect()[0]
pooled_nok_pct = 100.0 * pooled_nok[0] / pooled_nok[1]
print("Pooled margin, NOK filings only                       = %.4f%%" % pooled_nok_pct)
print("Difference                                            = %.4f pp"
      % (pooled_pct - pooled_nok_pct))

# Row share is the wrong denominator for a revenue-weighted metric: a handful of
# large foreign-currency subsidiaries could carry weight far beyond their count.
# Revenue share is what decides whether the exclusion is immaterial.
n_excluded = n_computed - nok.count()
rev_share = 100.0 * (1.0 - pooled_nok[1] / pooled[1])
print("Excluded from pooled figures: %d rows (%.3f%% of computed rows, "
      "%.3f%% of summed revenue)"
      % (n_excluded, 100.0 * n_excluded / n_computed, rev_share))

summary["margin"] = {
    "computed_rows": n_computed,
    "quantiles": dict(zip(labels, qs)),
    "min_pct": float(extremes[0]),
    "max_pct": float(extremes[1]),
    "mean_pct": float(extremes[2]),
    "pooled_pct": pooled_pct,
    "pooled_pct_nok_only": pooled_nok_pct,
    "pooled_currency_effect_pp": pooled_pct - pooled_nok_pct,
    "excluded_from_pooled_rows": n_excluded,
    "excluded_from_pooled_revenue_share_pct": rev_share,
}

Per-company operating margin (%), computed rows only:
  p0.1          -50338.89
  p1             -1974.82
  p5              -192.13
  p25               -4.13
  median             6.53
  p75               31.63
  p95               79.27
  p99               97.56
  p99.9            184.22
  min      -2731645700.00
  max          1800100.00
  mean          -11786.56   <- reported only to show why it is not used

Pooled margin  SUM(operating_income)/SUM(revenue)*100 = 5.5114%
Pooled margin, NOK filings only                       = 5.3088%
Difference                                            = 0.2025 pp
Excluded from pooled figures: 812 rows (0.277% of computed rows, 1.938% of summed revenue)


## Write

`coalesce(1)` into container-local scratch, then copy the single part file onto
the Dropbox-synced mount. At this row count the cost of collapsing to one
partition is negligible, and it gives PowerBI a plain file path with no
Folder/Combine step.

Scratch and deliverable deliberately live on different filesystems. See the
filesystem note at the top.

In [6]:
if os.path.exists(STAGING_DIR):
    shutil.rmtree(STAGING_DIR, ignore_errors=True)

curated.coalesce(1).write.mode("overwrite").parquet(STAGING_DIR)

parts = glob.glob(os.path.join(STAGING_DIR, "part-*.parquet"))
assert len(parts) == 1, "expected exactly one part file, got %d" % len(parts)

if os.path.exists(ANALYTICS_FILE):
    os.remove(ANALYTICS_FILE)

# copyfile rather than move: /tmp and the bind mount are different filesystems,
# and a copy leaves the source intact if the destination write is interrupted.
shutil.copyfile(parts[0], ANALYTICS_FILE)

# Cleanup is on container-local disk, but stays non-fatal either way: a failure
# here leaves scratch behind, it does not invalidate the deliverable.
shutil.rmtree(STAGING_DIR, ignore_errors=True)

summary["output"] = {"file": ANALYTICS_FILE,
                     "bytes": os.path.getsize(ANALYTICS_FILE),
                     "columns": len(curated.columns)}

print("Wrote %s  (%.1f MB, %d columns)"
      % (ANALYTICS_FILE, os.path.getsize(ANALYTICS_FILE) / 1024**2, len(curated.columns)))

Wrote /home/jovyan/data/parquet/analytics_company_financials.parquet  (91.1 MB, 61 columns)


## Verification

Re-read from disk rather than trusting the in-memory DataFrame, so a write that
silently dropped rows or columns is caught.

Two checks matter most. The recomputation check re-derives the margin from the
two stored components and requires agreement to 1e-9 relative, proving the
stored ratio and its numerator and denominator are mutually consistent — the
property PowerBI depends on when it computes pooled margins from components.
The balance sheet identity is the other. Total assets must equal total equity
plus liabilities in every filed statement; that is an accounting constraint,
not an assumption about this dataset.

A violation means either a source data problem or a flattening error here, and
both are worth knowing before the figures reach a dashboard.

In [7]:
out = spark.read.parquet(ANALYTICS_FILE)

n_disk = out.count()
print("Rows on disk: %d  %s" % (n_disk, "OK" if n_disk == n_rows else "MISMATCH"))
assert n_disk == n_rows

# Every null margin must carry a non-'computed' reason, and vice versa.
bad = out.filter(
    (F.col("operating_margin_pct").isNull() & (F.col("operating_margin_status") == "computed"))
    | (F.col("operating_margin_pct").isNotNull() & (F.col("operating_margin_status") != "computed"))
).count()
print("Margin/status contradictions: %d  %s" % (bad, "OK" if bad == 0 else "FAIL"))
assert bad == 0

# har_regnskap must be a real boolean everywhere. Before the coalesce in build()
# it was NULL for entities with no financial_data document, which made the check
# below evaluate to NULL and silently skip exactly those rows.
n_null_flag = out.filter(F.col("har_regnskap").isNull()).count()
print("har_regnskap nulls:           %d  %s"
      % (n_null_flag, "OK" if n_null_flag == 0 else "FAIL"))
assert n_null_flag == 0

# Every unfiled row must carry the no_filing status, and no filed row may.
mislabelled = out.filter(
    (~F.col("har_regnskap") & (F.col("operating_margin_status") != "no_filing"))
    | (F.col("har_regnskap") & (F.col("operating_margin_status") == "no_filing"))
).count()
print("no_filing mislabelled rows:   %d  %s" % (mislabelled, "OK" if mislabelled == 0 else "FAIL"))
assert mislabelled == 0

# Recompute from the stored components and compare.
recomputed = out.filter("operating_margin_status = 'computed'").withColumn(
    "delta",
    F.abs(F.col("operating_margin_pct") - (F.col("operating_income") / F.col("revenue")) * 100)
    / F.greatest(F.abs(F.col("operating_margin_pct")), F.lit(1.0)))
worst = recomputed.agg(F.max("delta")).collect()[0][0]
print("Worst relative recomputation error: %.3e  %s"
      % (worst, "OK" if worst < 1e-9 else "FAIL"))
assert worst < 1e-9

# No computed row may have a non-positive denominator.
n_nonpositive = out.filter(
    "operating_margin_status = 'computed' AND revenue <= 0").count()
print("No computed row has revenue <= 0  %s" % ("OK" if n_nonpositive == 0 else "FAIL"))
assert n_nonpositive == 0

# Balance sheet identity: assets = equity + liabilities. Reported rather than
# asserted, because a violation is a finding about the source, not a bug to
# abort on, and the report should carry the number either way.
bal = out.filter(F.col("sum_eiendeler").isNotNull()
                 & F.col("sum_egenkapital_gjeld").isNotNull())
n_bal = bal.count()
n_broken = bal.filter(
    F.abs(F.col("sum_eiendeler") - F.col("sum_egenkapital_gjeld")) > 0.5).count()
print("Balance identity: checked=%d  violations=%d (%.4f%%)"
      % (n_bal, n_broken, 100.0 * n_broken / n_bal if n_bal else 0))
balance = {"checked": n_bal, "violations": n_broken,
           "pct": 100.0 * n_broken / n_bal if n_bal else 0.0}

if n_broken:
    # A count alone cannot distinguish a rounding convention from a broken
    # statement. Absolute magnitude says how large the error is; relative
    # magnitude says whether it matters against the size of the balance sheet.
    # One krone off on a billion is a different finding from half the assets
    # missing, and only the relative figure separates them.
    broken = (bal.filter(F.abs(F.col("sum_eiendeler")
                               - F.col("sum_egenkapital_gjeld")) > 0.5)
              .withColumn("d_abs", F.abs(F.col("sum_eiendeler")
                                         - F.col("sum_egenkapital_gjeld")))
              .withColumn("d_rel", F.col("d_abs")
                          / F.greatest(F.abs(F.col("sum_eiendeler")), F.lit(1.0))))

    probs = [0.5, 0.9, 0.99]
    q_abs = broken.approxQuantile("d_abs", probs, 0.0)
    q_rel = broken.approxQuantile("d_rel", probs, 0.0)
    balance["discrepancy_abs_nok"] = dict(zip(["p50", "p90", "p99"], q_abs))
    balance["discrepancy_relative"] = dict(zip(["p50", "p90", "p99"], q_rel))

    print("  discrepancy NOK   p50=%.1f  p90=%.1f  p99=%.1f" % tuple(q_abs))
    print("  discrepancy rel   p50=%.4f  p90=%.4f  p99=%.4f" % tuple(q_rel))

    # A discrepancy under one krone in a thousand of total assets is a
    # presentation artefact; anything above that is a statement that does not
    # balance. Splitting on that threshold is what separates a finding from
    # source rounding, so every breakdown below uses the material subset only.
    MATERIAL_REL = 0.001
    trivial = broken.filter(F.col("d_rel") < MATERIAL_REL)
    material = broken.filter(F.col("d_rel") >= MATERIAL_REL).cache()

    n_trivial = trivial.count()
    n_material = material.count()
    balance["threshold_relative"] = MATERIAL_REL
    balance["violations_trivial"] = n_trivial
    balance["violations_material"] = n_material
    balance["violations_material_pct_of_checked"] = 100.0 * n_material / n_bal
    print("  trivial  (< %.1f%% of assets): %6d  (%.2f%% of violations)"
          % (100 * MATERIAL_REL, n_trivial, 100.0 * n_trivial / n_broken))
    print("  material (>= %.1f%% of assets): %6d  (%.4f%% of all filings)"
          % (100 * MATERIAL_REL, n_material, 100.0 * n_material / n_bal))

    # Counts alone cannot separate a high failure rate from a large population:
    # 44 IFRS violations and 15,243 under the ordinary rules are uninterpretable
    # without their denominators. Each breakdown is therefore joined to the base
    # population and reported as a rate.
    def rate_by(column):
        base = (bal.groupBy(column).agg(F.count(F.lit(1)).alias("filings")))
        hits = (material.groupBy(column).agg(F.count(F.lit(1)).alias("material")))
        rows = (base.join(hits, column, "left")
                .fillna(0, subset=["material"])
                .orderBy(F.desc("filings")).collect())
        out_d = {}
        for r in rows:
            pct = 100.0 * r["material"] / r["filings"] if r["filings"] else 0.0
            out_d[str(r[column])] = {"filings": r["filings"],
                                     "material": r["material"], "pct": pct}
            print("    %-32s %8d filings %7d material %7.3f%%"
                  % (str(r[column]), r["filings"], r["material"], pct))
        return out_d

    print("  material violation rate by regnskapsregler:")
    balance["material_by_regnskapsregler"] = rate_by("regnskapsregler")
    print("  material violation rate by smaa_foretak:")
    balance["material_by_smaa_foretak"] = rate_by("smaa_foretak")

    # The size pattern in the raw counts may be an artefact of the absolute
    # 0.5 NOK tolerance: a larger balance sheet produces a larger rounding
    # residue and trips an absolute threshold more easily. Comparing the two
    # rates on the same subset is what decides whether the pattern is real.
    print("  same rate over ALL violations, for comparison with the above:")
    all_by_smaa = (bal.groupBy("smaa_foretak").agg(F.count(F.lit(1)).alias("filings"))
                   .join(broken.groupBy("smaa_foretak")
                         .agg(F.count(F.lit(1)).alias("any")), "smaa_foretak", "left")
                   .fillna(0, subset=["any"]).collect())
    balance["all_violations_by_smaa_foretak"] = {}
    for r in all_by_smaa:
        pct = 100.0 * r["any"] / r["filings"] if r["filings"] else 0.0
        balance["all_violations_by_smaa_foretak"][str(r["smaa_foretak"])] = {
            "filings": r["filings"], "any": r["any"], "pct": pct}
        print("    smaa_foretak=%-6s %8d filings %7d any %7.3f%%"
              % (str(r["smaa_foretak"]), r["filings"], r["any"], pct))

    # Zero-asset statements are the extreme tail: total assets at or near zero
    # while the equity and liability side is populated. These cannot be rounding.
    n_zero_assets = material.filter(F.abs(F.col("sum_eiendeler")) < 0.5).count()
    balance["material_with_zero_assets"] = n_zero_assets
    print("  of the material cases, %d report total assets of zero" % n_zero_assets)

    material.orderBy(F.desc("d_rel")).select(
        "organisasjonsnummer", "navn", "organisasjonsform_kode",
        "sum_eiendeler", "sum_egenkapital_gjeld", "d_abs", "d_rel").show(5, truncate=28)
    material.unpersist()

summary["verification"] = {
    "rows_on_disk": n_disk,
    "margin_status_contradictions": bad,
    "har_regnskap_nulls": n_null_flag,
    "no_filing_mislabelled": mislabelled,
    "worst_relative_recomputation_error": float(worst),
    "computed_rows_with_nonpositive_revenue": n_nonpositive,
    "balance_identity": balance,
}

print("\nSchema:")
out.printSchema()

Rows on disk: 1171373  OK
Margin/status contradictions: 0  OK
har_regnskap nulls:           0  OK
no_filing mislabelled rows:   0  OK
Worst relative recomputation error: 0.000e+00  OK
No computed row has revenue <= 0  OK
Balance identity: checked=444645  violations=15462 (3.4774%)
  discrepancy NOK   p50=1.0  p90=2000.0  p99=1159611.0
  discrepancy rel   p50=0.0000  p90=0.0061  p99=30000.0000
  trivial  (< 0.1% of assets):  13701  (88.61% of violations)
  material (>= 0.1% of assets):   1761  (0.3960% of all filings)
  material violation rate by regnskapsregler:
    regnskapslovenAlminneligRegler     442947 filings    1748 material   0.395%
    forenkletAnvendelseIFRS              1476 filings       9 material   0.610%
    IFRS                                  222 filings       4 material   1.802%
  material violation rate by smaa_foretak:
    True                               434467 filings    1717 material   0.395%
    False                               10178 filings      44 materi

## Why a large share of filings have no margin

`revenue_missing` is a substantial status class among filed rows, so it needs
anexplanation rather than a footnote. Two competing hypotheses, both
testablefrom columns already in the table:1. **Layout.** The API omits the
`driftsinntekter` node for statement layouts that do not have one — banks and
insurers use a different income statement. If so, the missing cases concentrate
in particular `oppstillingsplan` values and the rest of the statement is
present.2. **Dormant and holding companies.** Income is financial rather than
operating, so there is no operating revenue to report. If so, the missing cases
spread across layouts and correlate with legal form. The second block below
discriminates between them: if `operating_costs` and the balance sheet are
present while revenue is not, the node is layout-specific rather than the
filing being empty.

Restricted to filed rows throughout — `no_filing` rows are a separate class and
would swamp the comparison.

In [8]:
filed_only = out.filter("har_regnskap")

# Hypothesis 1: the missing revenue tracks the statement layout.
layout = (filed_only.groupBy("oppstillingsplan")
          .agg(F.count("*").alias("n"),
               F.sum(F.when(F.col("operating_margin_status") == "revenue_missing", 1)
                      .otherwise(0)).alias("revenue_missing"),
               F.sum(F.when(F.col("operating_margin_status") == "revenue_zero", 1)
                      .otherwise(0)).alias("revenue_zero"))
          .orderBy(F.desc("n")).collect())

print("%-24s %9s %9s %8s %9s %7s" % ("oppstillingsplan", "n", "missing", "pct", "zero", "pct"))
print("-" * 72)
by_layout = {}
for r in layout:
    key = r["oppstillingsplan"] if r["oppstillingsplan"] is not None else "(null)"
    by_layout[key] = {"n": r["n"], "revenue_missing": r["revenue_missing"],
                      "revenue_zero": r["revenue_zero"],
                      "pct_missing": 100.0 * r["revenue_missing"] / r["n"],
                      "pct_zero": 100.0 * r["revenue_zero"] / r["n"]}
    v = by_layout[key]
    print("%-24s %9d %9d %7.2f%% %9d %6.2f%%"
          % (key, v["n"], v["revenue_missing"], v["pct_missing"],
             v["revenue_zero"], v["pct_zero"]))

# Is the whole statement absent, or only the revenue line? If costs and the
# balance sheet are present while revenue is not, the node is layout-specific
# rather than the filing being empty.
missing = filed_only.filter("operating_margin_status = 'revenue_missing'")
n_missing = missing.count()
print("\nWithin the %d revenue_missing filings, how much else is present:" % n_missing)
completeness = {}
for col in ["operating_costs", "operating_income", "sum_eiendeler", "sum_egenkapital"]:
    present = missing.filter(F.col(col).isNotNull()).count()
    completeness[col] = {"present": present, "pct": 100.0 * present / n_missing}
    print("  %-18s present in %7d  (%5.2f%%)" % (col, present, completeness[col]["pct"]))

# Hypothesis 2: these are dormant and holding companies, so the missing cases
# should track legal form and the parent-company flag rather than layout.
print("\nLegal forms carrying the missing cases:")
by_form = {r["organisasjonsform_kode"]: r["count"] for r in
           missing.groupBy("organisasjonsform_kode").count()
           .orderBy(F.desc("count")).limit(10).collect()}
for k, v in by_form.items():
    print("  %-8s %8d  (%5.2f%%)" % (k, v, 100.0 * v / n_missing))

print("\nStatus mix for parent companies vs the rest:")
by_morselskap = {}
for r in filed_only.groupBy("morselskap", "operating_margin_status").count().collect():
    by_morselskap.setdefault(str(r["morselskap"]), {})[r["operating_margin_status"]] = r["count"]
for flag in sorted(by_morselskap):
    total = sum(by_morselskap[flag].values())
    print("  morselskap=%s  (n=%d)" % (flag, total))
    for st in sorted(by_morselskap[flag], key=lambda k: -by_morselskap[flag][k]):
        print("      %-18s %8d  %6.2f%%"
              % (st, by_morselskap[flag][st], 100.0 * by_morselskap[flag][st] / total))

# Are revenue_missing and revenue_zero the same economic fact, differing only
# in whether Brreg wrote a 0 or omitted the node? If operating revenue really is
# zero then driftsresultat is just the negated cost line, so operating_income
# should equal -operating_costs. Agreement here would let the two classes be
# reported as one "no operating revenue" figure rather than two separate gaps.
zero_test = filed_only.filter(
    "operating_margin_status = 'revenue_missing' "
    "AND operating_income IS NOT NULL AND operating_costs IS NOT NULL")
n_zero_test = zero_test.count()
n_consistent = zero_test.filter(
    F.abs(F.col("operating_income") + F.col("operating_costs")) < 0.5).count()
pct_consistent = 100.0 * n_consistent / n_zero_test if n_zero_test else 0.0
print("\nZero-revenue test on revenue_missing rows with both lines present:")
print("  operating_income == -operating_costs in %d of %d  (%.2f%%)"
      % (n_consistent, n_zero_test, pct_consistent))

# The same identity must already hold on revenue_zero rows, where revenue is
# explicitly 0. Running it there too gives a control: if it holds at a similar
# rate, the comparison is meaningful; if it fails there, the test itself is
# measuring something other than what it claims.
ctrl = filed_only.filter(
    "operating_margin_status = 'revenue_zero' "
    "AND operating_income IS NOT NULL AND operating_costs IS NOT NULL")
n_ctrl = ctrl.count()
n_ctrl_ok = ctrl.filter(
    F.abs(F.col("operating_income") + F.col("operating_costs")) < 0.5).count()
pct_ctrl = 100.0 * n_ctrl_ok / n_ctrl if n_ctrl else 0.0
print("  control, same test on revenue_zero: %d of %d  (%.2f%%)"
      % (n_ctrl_ok, n_ctrl, pct_ctrl))

n_no_revenue = status_counts.get("revenue_missing", 0) + status_counts.get("revenue_zero", 0)
print("  combined no-operating-revenue filings: %d  (%.2f%% of %d filed)"
      % (n_no_revenue, 100.0 * n_no_revenue / n_filed, n_filed))

summary["zero_revenue_test"] = {
    "revenue_missing_tested": n_zero_test,
    "revenue_missing_consistent": n_consistent,
    "revenue_missing_pct": pct_consistent,
    "revenue_zero_tested": n_ctrl,
    "revenue_zero_consistent": n_ctrl_ok,
    "revenue_zero_pct": pct_ctrl,
    "combined_no_revenue_filings": n_no_revenue,
    "combined_pct_of_filed": 100.0 * n_no_revenue / n_filed,
}

summary["revenue_missing_diagnostic"] = {
    "n_revenue_missing": n_missing,
    "by_oppstillingsplan": by_layout,
    "other_fields_present": completeness,
    "top_legal_forms": by_form,
    "status_by_morselskap": by_morselskap,
}

oppstillingsplan                 n   missing      pct      zero     pct
------------------------------------------------------------------------
store                       444645     91026   20.47%     59206  13.32%

Within the 91026 revenue_missing filings, how much else is present:
  operating_costs    present in   83543  (91.78%)
  operating_income   present in   84024  (92.31%)
  sum_eiendeler      present in   91026  (100.00%)
  sum_egenkapital    present in   89508  (98.33%)

Legal forms carrying the missing cases:
  AS          87486  (96.11%)
  STI          1518  ( 1.67%)
  NUF           489  ( 0.54%)
  BRL           425  ( 0.47%)
  ESEK          246  ( 0.27%)
  ANS           192  ( 0.21%)
  ENK           169  ( 0.19%)
  DA            143  ( 0.16%)
  FLI           127  ( 0.14%)
  SA             65  ( 0.07%)

Status mix for parent companies vs the rest:
  morselskap=False  (n=387643)
      computed             268089   69.16%
      revenue_missing       69818   18.01%
      rev

## Scalability experiment

The same `build` function run at `local[1]`, `local[2]`, `local[4]`, `local[8]`
and `local[12]` on a 12-core host, ending in a write so shuffle and output cost
are included rather than only the scan.

Method matches `Benchmark_engines.ipynb`: one discarded warm-up run per
setting, then three timed runs, median reported.

Nothing is cached during the timed runs, so each measures a cold build from the
Parquet mirror.

`spark.driver.memory` is fixed at 8 GB by `spark-defaults.conf` and is applied
at JVM launch, so it cannot change between settings — the heap is held constant
and thread count is the only variable. That is the controlled comparison we
want. `spark.master`, by contrast, is read when the `SparkContext` is created,
so stopping and recreating the session is enough to change it.

Output goes to `/tmp` inside the container, not the Dropbox-synced `data/`.
**This cell stops the session.** Everything above that depends on `spark` or
`curated` must have run already; the summary cell below uses only the recorded
dict.

**If a session fails to restart**, restart the kernel, re-run the setup cell
and run this cell alone. Recreating a `SparkContext` in a live Python process
isreliable in local mode but is not something Spark guarantees.

In [9]:
THREAD_COUNTS = [1, 2, 4, 8, 12]
REPEATS = 3

# Free the cached build before timing; a cached DataFrame from the earlier
# session would make the first setting look artificially fast.
curated.unpersist()
spark.stop()

scal = {}

for n_threads in THREAD_COUNTS:
    session = (SparkSession.builder
               .appName("group13_scalability_local%d" % n_threads)
               .master("local[%d]" % n_threads)
               .getOrCreate())

    target = os.path.join(SCRATCH_DIR, "local%d" % n_threads)

    def run(session=session, target=target):
        build(session).coalesce(1).write.mode("overwrite").parquet(target)

    try:
        run()                                   # warm-up, discarded
        times = []
        for _ in range(REPEATS):
            t0 = time.perf_counter()
            run()
            times.append(time.perf_counter() - t0)
        scal["local[%d]" % n_threads] = {"times": times, "error": None}
        print("local[%-2d]  median %6.2fs   min %6.2fs   max %6.2fs"
              % (n_threads, statistics.median(times), min(times), max(times)))
    except Exception as exc:                    # noqa: BLE001
        scal["local[%d]" % n_threads] = {"times": [], "error": "%s: %s" % (type(exc).__name__, exc)}
        print("local[%-2d]  DID NOT COMPLETE  %s" % (n_threads, exc))

    session.stop()

if os.path.exists(SCRATCH_DIR):
    shutil.rmtree(SCRATCH_DIR, ignore_errors=True)

summary["scalability"] = {"thread_counts": THREAD_COUNTS, "repeats": REPEATS,
                          "seconds": scal}

local[1 ]  median  20.52s   min  20.37s   max  20.89s
local[2 ]  median  15.45s   min  15.20s   max  15.67s
local[4 ]  median  14.26s   min  13.22s   max  15.09s
local[8 ]  median  13.51s   min  12.84s   max  14.96s
local[12]  median  15.27s   min  14.80s   max  15.62s


## Speedup

Speedup relative to `local[1]`, against the ideal linear line. The gap between
them is the interesting quantity: it is where Amdahl's law, the non-splittable
parts of the job and the single-partition write become visible.

In [10]:
baseline = statistics.median(scal["local[1]"]["times"]) if scal["local[1]"]["times"] else None

derived = {}
print("%-10s %10s %10s %10s %12s" % ("setting", "median s", "speedup", "ideal", "efficiency"))
print("-" * 56)
for n_threads in THREAD_COUNTS:
    key = "local[%d]" % n_threads
    times = scal[key]["times"]
    if not times or baseline is None:
        print("%-10s %10s" % (key, "n/a"))
        continue

    med = statistics.median(times)
    speedup = baseline / med
    derived[key] = {"median_s": med, "speedup": speedup,
                    "efficiency": speedup / n_threads}
    print("%-10s %10.2f %10.2fx %9dx %11.0f%%"
          % (key, med, speedup, n_threads, 100.0 * speedup / n_threads))

summary["scalability"]["derived"] = derived
summary["scalability"]["fastest_setting"] = min(
    derived, key=lambda k: derived[k]["median_s"]) if derived else None

setting      median s    speedup      ideal   efficiency
--------------------------------------------------------
local[1]        20.52       1.00x         1x         100%
local[2]        15.45       1.33x         2x          66%
local[4]        14.26       1.44x         4x          36%
local[8]        13.51       1.52x         8x          19%
local[12]       15.27       1.34x        12x          11%


## Persist the results

Written next to the benchmark output so the report can cite figures from a file
rather than from a notebook that has since been re-run.

In [11]:
# Nothing is assembled here. Every value was recorded by the cell that measured
# it, so this only serialises what the run already produced. Missing keys mean
# a cell was skipped, which is information worth preserving rather than hiding.
path = os.path.join(DATA_DIR, "analytics_build_summary.json")
with open(path, "w") as fh:
    json.dump(summary, fh, indent=2)

print("Wrote", path)
print("Sections recorded:")
for k in summary:
    print("  ", k)

Wrote /home/jovyan/data/analytics_build_summary.json
Sections recorded:
   built_at
   spark_version
   spark_master
   driver_max_heap_gb
   default_parallelism
   cpu_cores
   date_parse_check
   cardinality
   fetch_status_counts
   status_counts
   coverage_by_legal_form
   stiftelsesdato_missing_total
   stiftelsesdato_coverage_by_legal_form
   flag_counts
   valuta_counts
   regnskapstype_counts
   margin
   output
   verification
   zero_revenue_test
   revenue_missing_diagnostic
   scalability


## What PowerBI connects to

Get Data, Parquet, then `data/parquet/analytics_company_financials.parquet`.One
table, one row per registered entity. Group by `naeringskode1_kode` and
`kommunenummer` to reproduce the benchmark's W4 aggregate, or by
`organisasjonsform_kode`, `stiftelsesaar`, `regnskapsaar`, `konkurs` or
anything else — the table was deliberately left at row level so that choice
stays open.

Two rules for measures:- **Ratios come from summed components, never from
averaged ratios.** Pooled operating margin is `DIVIDE(SUM(operating_income),
SUM(revenue))`.

Equity ratio is `DIVIDE(SUM(sum_egenkapital), SUM(sum_eiendeler))`. Averaging
`operating_margin_pct` across a group gives a different and generally wrong
answer, because the mean of ratios is not the ratio of sums.- **Filter to
`currency_comparable` for any measure that sums money.** Ratios and
distributions may use every row; totals may not, because summing across
currencies adds unlike units.